# 通过时间反向传播
因为序列比较长，可能存在长程依赖， 比如1000个字符的序列， 所以需要1000个矩阵的乘法才可能得到梯度，中间容易出现梯度消失或者梯度爆炸

每个时间步的隐状态的方程式如下所示：
$$h_t = f(x_t, h_{t-1}, w_h), \quad o_t = g(h_t, w_o)$$

这里有一个链循环计算彼此的依赖：$\{ \dots, (x_{t-1}, h_{t-1}, o_{t-1}), (x_{t}, h_{t}, o_{t}), \dots\}$

在前向传播的过程中， 一次一个时间步的遍历三元组$(x_t, h_t, o_t)$

最后通过一个目标函数计算T个时间步内输出和对应标签之间的差值，作为损失函数：
$$L(x_1, ..., x_T, y_1, ..., y_T, w_h, w_o) = \frac{1}{T} \sum_{t=1}^T l(y_t, o_t)$$

## 反向传播的链式法则展开

计算目标是找到损失 $L$ 对隐藏层权重 $w_h$ 的梯度：
$$\frac{\partial L}{\partial w_h} = \frac{1}{T} \sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_h} = \frac{1}{T} \sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_o)}{\partial h_t} \frac{\partial h_t}{\partial w_h}$$

这里的$o, g, h$对应的是正向传播过程中隐藏时间步和输出的状态方程

最棘手的是最后一项 $\frac{\partial h_t}{\partial w_h}$ ，因为 $h_t$ 的计算中不但直接用到了 $w_h$ ，还用到了上一时刻的 $h_{t-1}$ ，而 $h_{t-1}$ 里面又包含了 $w_h$。

（这里的$w_h$是隐藏层的参数的数量）

$$\frac{\partial h_t}{\partial w_h} = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h} + \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_h}$$

上面我们得到了一个递归的梯度表达式，下面我们需要进行一些化简：
令 $a_t = \frac{\partial h_t}{\partial w_h}$ ， $b_t = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h}$ ， $c_t = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial h_{t-1}}$

上式可以先写成：

我们先得到原始的递归公式：$$a_t = b_t + c_t a_{t-1}$$

依次代入时间步有：$$a_1 = b_1$$

$$a_2 = b_2 + c_2 a_1 = b_2 + c_2 b_1$$

$$a_3 = b_3 + c_3 a_2 = b_3 + c_3 (b_2 + c_2 b_1) = b_3 + c_3 b_2 + c_3 c_2 b_1$$

所以我们可以得到这个数列的求和通式：
$$a_t = b_t + \sum_{i=1}^{t-1} \left( \prod_{j=i+1}^t c_j \right) b_i$$

代入具体数值之后可以得到求和公式:
$$\frac{\partial h_t}{\partial w_h} = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h} + \sum_{i=1}^{t-1} \left( \prod_{j=i+1}^t \frac{\partial f(x_j, h_{j-1}, w_h)}{\partial h_{j-1}} \right) \frac{\partial f(x_i, h_{i-1}, w_h)}{\partial w_h}$$